# Step 1 : Set up Your Environment 

In [5]:
!pip install pandas numpy scikit-learn nltk

# Step 2 : Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Step 3 : Load Dataset

In [4]:
data = pd.read_csv('spam.csv', encoding='latin-1')
data = data[['v1', 'v2']]
data.columns = ['label', 'message'] 

In [6]:
data

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will Ì_ b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [8]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    5572 non-null   object
 1   message  5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [10]:
data.describe()

,label,message
count,5572,5572
unique,2,5169
top,ham,"Sorry, I'll call later"
freq,4825,30


# Step 4 : Data Preprocessing

* **Convert labels to binary values (e.g., 1 for spam and 0 for ham).**
* **Clean text data by removing stopwords and punctuation.**

In [13]:
# Check for missing values in 'label' and 'message' columns
print("Missing values in 'label':", data['label'].isnull().sum())
print("Missing values in 'message':", data['message'].isnull().sum())

Missing values in 'label': 0
Missing values in 'message': 0


In [15]:
# Drop rows with NaN values in either 'label' or 'message'
data = data.dropna(subset=['label', 'message'])

In [17]:
# Verify all missing values were removed from the original dataset
assert data.isnull().sum().sum() == 0, "Initial data still contains NaN values."

In [19]:
# Convert 'label' to binary format
data['label'] = data['label'].map({'ham': 0, 'spam': 1})
# Verify there are no NaNs in 'label' after mapping
assert data['label'].isnull().sum() == 0, "Labels still contain NaN values after mapping."

In [21]:
# Fill any NaN values in 'message' with an empty string, if needed
data['message'] = data['message'].fillna('')

In [21]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\bharg\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [23]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\bharg\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [25]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bharg\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [27]:
data['label'] = data['label'].map({'ham': 0, 'spam': 1})

nltk.download('stopwords')
nltk.download('punkt')
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Tokenize
    words = word_tokenize(text.lower())
    # Remove stopwords and non-alphabetic words
    words = [word for word in words if word.isalpha() and word not in stop_words]
    return ' '.join(words)

data['message'] = data['message'].apply(preprocess_text) 

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bharg\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\bharg\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [23]:
# Verify no missing values exist in 'message' after preprocessing
assert data['message'].isnull().sum() == 0, "Messages still contain NaN values after preprocessing."

# Step 5 : Feature Extraction 
* **Use TF-IDF (Term Frequency-Inverse Document Frequency) to convert text data into numerical features.**

In [25]:
vectorizer = TfidfVectorizer(max_features=3000)
X = vectorizer.fit_transform(data['message'])
y = data['label']

In [27]:
# Verify no NaN values exist in the TF-IDF matrix or target variable
if np.any(np.isnan(X.toarray())):
    raise ValueError("The TF-IDF matrix X contains NaN values.")
if np.any(np.isnan(y)):
    raise ValueError("The target variable y contains NaN values.")

# Step 6 : Train - Test Split
* **Split the dataset into training and test sets to evaluate the model.**

In [29]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 7 : Train the Model
* **Use a classifier such as Multinomial Naive Bayes, which is effective for text classification.**

In [31]:
model = MultinomialNB()
model.fit(X_train, y_train) 

MultinomialNB()

# Step 8 : Evalute the Model

In [33]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)
print(f"Accuracy: {accuracy}")
print(report)

Accuracy: 0.9757847533632287
              precision    recall  f1-score   support

           0       0.97      1.00      0.99       965
           1       1.00      0.82      0.90       150

    accuracy                           0.98      1115
   macro avg       0.99      0.91      0.94      1115
weighted avg       0.98      0.98      0.97      1115



In [35]:
y_pred

array([0, 0, 0, ..., 0, 0, 1], dtype=int64)

# Step 9 : Test the Model with New Data

In [39]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

def preprocess_text(text):
    # Convert text to lowercase
    text = text.lower()
    # Remove any special characters or digits
    text = re.sub(r'[^a-z\s]', '', text)
    return text

def predict_spam(text):
    text = preprocess_text(text)
    text_vec = vectorizer.transform([text])
    prediction = model.predict(text_vec)
    return "Spam" if prediction[0] == 1 else "Ham"

# Test with a new email
print(predict_spam("Congratulations! You've won a $1000 gift card. Claim now!"))

Spam


# Step 10: Fine - tune and improve

* **Experiment with different models such as SVM, Logistic Regression, or Decision Trees.**
* **Try adding additional preprocessing steps like lemmatization, stemming, or using Word2Vec embeddings for text representation.**

**This notebook will give you a basic spam detection application. With further tuning and a larger dataset, you can improve the accuracy and reliability of your model.**